### 0. Database Connection Setup
Initializes the Ibis connection to the DuckDB database to load the fixed and yearly FAME tables.

In [3]:
import ibis
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="build")
out_file = dirs.output_dir / "duckdb_tables.md"
con = ibis.duckdb.connect(str(dirs.db_path))
interesting_tables = ["fame_fixed", "fame_derived", "fame_yearly", "lars_fixed", "lars_yearly"]
existing_tables = con.list_tables()
intersection = set(interesting_tables).intersection(existing_tables)
print(existing_tables)

with open(out_file, "w") as f:
    f.write("# Tables in DuckDB database\n\n")
    for table in intersection:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute():,}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).head().execute()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

['fame_fixed', 'fame_fixed_filtered', 'fame_yearly', 'fame_yearly_filtered', 'lars_fixed', 'lars_yearly']
✅ Successfully listed tables and their heads in: C:\Users\lazyst\Files\ucl\Dissertation\build\output\duckdb_tables.md


In [4]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="build")
con = ibis.connect(str(dirs.db_path))

fame_yearly = con.table("fame_yearly_filtered")
fame_fixed = con.table("fame_fixed_filtered")
print("✅ Connected to database.")

✅ Connected to database.


### 1. Small Companies Exclusion
Verify that no firm-year entries have fewer than 10 employees or missing employment data.

In [5]:
invalid_employees = fame_yearly.filter(
    fame_yearly.employees.isnull() | (fame_yearly.employees < 10)
).count().execute()

try:
    assert invalid_employees == 0, f"Found {invalid_employees:,} invalid employee entries."
    print("✅ Employee threshold test passed.")
except AssertionError as e:
    print(f"❌ Employee threshold test failed: {e}")
    sample_invalid = fame_yearly.filter(
        fame_yearly.employees.isnull() | (fame_yearly.employees < 10)
    ).sample(100 / invalid_employees).execute()
    display(sample_invalid)

✅ Employee threshold test passed.


### 2. UK Registered Numbers Only
Verify that non-UK companies (e.g., prefixes '#', 'IE', 'GI') have been successfully dropped.

In [6]:
uk_prefixes = [
    '',   # England & Wales (pure numbers)
    'NI', # Northern Ireland Company (post-partition)
    'SC', # Scottish Company
    'OC', # Limited Liability Partnership - LLP (England & Wales)
    'SO', # Limited Liability Partnership - LLP (Scotland)
    'NC', # Limited Liability Partnership - LLP (Northern Ireland)
    'LP', # Limited Partnership (England & Wales)
    'SL', # Limited Partnership (Scotland)
    'ZC', # Unregistered Companies (Section 1043) for England & Wales
    'SZ', # Scottish Unregistered Companies (Section 1043)
    'SG', # Scottish Qualifying Partnership
    'CE', # Charitable Incorporated Organisation (England & Wales)
    'CS', # Scottish Charitable Incorporated Organisation
    'R',  # Older Northern Ireland company (no longer issued)
    'IP'  # Industrial and Provident Societies (cooperatives)
]
non_uk_prefixes = [
    'IE', # Ireland
    'JE', # Jersey
    'IM', # Isle of Man
    'GG', # Guernsey
    'GI', # Gibraltar,
    'SE'  # Société Européenne (European Company),
    '#',  # Foreign legal entites that are traded on LSEG
]

non_uk_regex = r'^(IE|JE|IM|GG|GI|SE|#)'

foreign_fixed = fame_fixed.filter(
    fame_fixed.registered_number.re_extract(non_uk_regex, 1).is_not_null()
).count().execute()
foreign_yearly = fame_yearly.filter(
    fame_yearly.registered_number.re_extract(non_uk_regex, 1).is_not_null()
).count().execute()

assert foreign_fixed == 0, f"Found {foreign_fixed} foreign firm records."
assert foreign_yearly == 0, f"Found {foreign_yearly} foreign firm records."
print("✅ UK-only test passed.")

AttributeError: 'StringColumn' object has no attribute 'is_not_null'

### 3. Firm Age Check
Verify that no firm age is recorded as negative (e.g., replacing -1 with 0). *Note: Runs if column is present.*

In [ ]:
if 'firm_age' in fame_yearly.columns:
    invalid_age = fame_yearly.filter(fame_yearly.firm_age < 0).count().execute()
    assert invalid_age == 0, f"Found {invalid_age} rows with negative firm age."
    print("✅ Firm age test passed.")
else:
    print("⚠️ firm_age column not present in schema. Skipping test.")

### 4. Duplicate Records Check
Verify that each `registered_number` has strictly one record per `year` in the yearly panel.

In [11]:
duplicates = fame_yearly.group_by(['registered_number', 'year']) \
    .aggregate(row_count=fame_yearly.count()) \
    .filter(ibis._.row_count > 1).count().execute()

try:
    assert duplicates == 0, f"Found {duplicates} duplicate firm-year records."
    print("✅ Firm-year duplicate test passed.")
except AssertionError as e:
    print(f"❌ Firm-year duplicate test failed: {e}")
    sample_duplicates = fame_yearly.group_by(['registered_number', 'year']) \
        .aggregate(row_count=fame_yearly.count()) \
        .filter(ibis._.row_count > 1) \
        .sample(100 / duplicates).execute()
    display(sample_duplicates)

✅ Firm-year duplicate test passed.


### 5. Same Company Name Check
Verify that no two separate `registered_number`s share the exact same `company_name`.

In [15]:
same_name = fame_fixed.group_by('company_name') \
    .aggregate(id_count=fame_fixed.registered_number.nunique()) \
    .filter(ibis._.id_count > 1).count().execute()

try:
    assert same_name == 0, f"Found {same_name} company names mapped to multiple registered numbers."
    print("✅ Shared company name test passed.")
except AssertionError as e:
    print(f"❌ Shared company name test failed: {e}")
    sample_same_name = fame_fixed.group_by('company_name') \
        .aggregate(id_count=fame_fixed.registered_number.nunique()) \
        .filter(ibis._.id_count > 1) \
        .inner_join(fame_fixed, ['company_name']) \
        .sample(100 / same_name if same_name >= 100 else 1).execute()
    display(sample_same_name)

❌ Shared company name test failed: Found 68 company names mapped to multiple registered numbers.


,company_name,id_count,registered_number,ticker_symbol,ro_address,ro_address_line_1,ro_address_line_2,ro_address_line_3,ro_address_line_4,ro_address_line_5,...,branch_name,primary_uk_sic_2007_code,primary_uk_sic_2007_description,latest_accounts_date,no_of_available_years,guo,guo_nb,entity_type,industry_codes,file_codes
0,PASSION HOLDINGS LIMITED,2,10418071,NaN,"33-34 Rathbone Place, London, W1T 1JN",33-34 Rathbone Place,NaN,NaN,NaN,NaN,...,PASSION HOLDINGS LIMITED,59113,Television programme production activities,2024-03-31,7,MRS MARIE RUHEMANN,39.0,Controlled subs.,59,18_17
1,VALENTINO ENGLAND LIMITED,2,05895246,NaN,"16 Old Bailey, London, EC4M 7EG",16 Old Bailey,NaN,NaN,NaN,NaN,...,VALENTINO ENGLAND LIMITED,82990,Other business support service activities n.e.c.,2013-12-31,7,NaN,0.0,Single location,82,10_05
2,HWS HOLDINGS LIMITED,2,09745689,NaN,"Fourth Floor Abbots House, Abbey Street, Readi...",Fourth Floor Abbots House,Abbey Street,NaN,NaN,NaN,...,NaN,82990,Other business support service activities n.e.c.,2023-12-31,9,TIDE UK HOLDINGS LLC,8.0,Controlled subs.,82,10_30
3,HOPKINS ARCHITECTS LIMITED,2,11779559,NaN,"27 Broadley Terrace, London, NW1 6LG",27 Broadley Terrace,NaN,NaN,NaN,NaN,...,HOPKINS ARCHITECTS LIMITED,71111,Architectural activities,2024-03-31,6,HOPKINS ARCHITECTS EMPLOYEE TRUSTEE LIMITED,2.0,Controlled subs.,71,18_27
4,HARROGATE V.E. LIMITED,2,13861617,NaN,"Vision Express, Ruddington Fields Business Par...",Vision Express,Ruddington Fields Business Park,Ruddington,NaN,NaN,...,HARROGATE V.E. LIMITED,96090,Other personal service activities n.e.c.,2023-12-31,2,ESSILORLUXOTTICA,994.0,Controlled subs.,96,17_32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,ROYDON HOLDINGS LIMITED,2,03599085,NaN,"Units 1-3 Junction Eco Park, Rake Lane, Swinto...",Units 1-3 Junction Eco Park,Rake Lane,Swinton,NaN,NaN,...,ROYDON HOLDINGS LIMITED,20160,Manufacture of plastics in primary forms,2012-10-31,14,NaN,0.0,Single location,20,22_11
134,LEONARDO LIMITED,2,05360430,NaN,"Lysander Road, Yeovil, Somerset, BA20 2YB",Lysander Road,NaN,NaN,NaN,NaN,...,LEONARDO LIMITED,70100,Activities of head offices,2023-12-31,19,LEONARDO S.P.A.,340.0,Controlled subs.,70,16_57 1
135,VIANET GROUP PLC,2,05345684,VNET,"1 Surtees Way, Surtees Business Park, Stockton...",1 Surtees Way,Surtees Business Park,NaN,NaN,NaN,...,VIANET GROUP PLC,96090,Other personal service activities n.e.c.,2024-03-31,19,VIANET GROUP PLC,7.0,GUO,96,17_32
136,CARE HOLDINGS LIMITED,2,06844795,NaN,"2 The Calls, Leeds, West Yorkshire, LS2 7JU",2 The Calls,NaN,NaN,NaN,NaN,...,CARE HOLDINGS LIMITED,87900,Other residential care activities,2023-12-31,15,NaN,NaN,NaN,"64,87","12_48,19_22"


### 6. Subsidiaries Check
Verify that no firm in the dataset is a subsidiary of another firm in the dataset (matching `guo` to `company_name`).

In [16]:
t1 = fame_fixed.alias('t1')
t2 = fame_fixed.alias('t2')

subsidiaries = t1.inner_join(
    t2, t1.guo == t2.company_name
).filter(t1.registered_number != t2.registered_number).count().execute()

assert subsidiaries == 0, f"Found {subsidiaries} firms that are subsidiaries of others in the dataset."
print("✅ Subsidiaries exclusion test passed.")

AssertionError: Found 17630 firms that are subsidiaries of others in the dataset.

### 7. Account Balancing Check
Verify the core accounting identity: Total Assets must equal Liabilities + Shareholders' Funds (allowing £2,000 margin of error).

In [ ]:
imbalance = fame_yearly.filter(
    abs(fame_yearly.total_assets - (fame_yearly.liabilities + fame_yearly.shareholders_funds)) > 2000
).count().execute()

assert imbalance == 0, f"Found {imbalance} rows failing the account balancing check."
print("✅ Account balancing test passed.")

### 8. Outlier Validation (Flagged but Kept)
Verify that negative assets are dropped, but extreme values (negative turnover, high turnover/assets) exist and are handled gracefully.

In [ ]:
negative_assets = fame_yearly.filter(fame_yearly.total_assets < 0).count().execute()
assert negative_assets == 0, "Negative total assets were not properly dropped."

negative_turnover = fame_yearly.filter(fame_yearly.turnover < 0).count().execute()
high_ratio = fame_yearly.filter((fame_yearly.turnover / fame_yearly.total_assets) > 50).count().execute()

print("✅ Negative assets properly dropped.")
print(f"ℹ️ Validated retention of negative turnover rows: {negative_turnover}")
print(f"ℹ️ Validated retention of high turnover/assets rows: {high_ratio}")

### 9. Deflation Check (Conceptual)
Verify that the yearly monetary columns are typed correctly as floats, indicating successful execution of the deflation pipeline.

In [ ]:
assert fame_yearly.turnover.type().is_float64(), "Turnover is not correctly typed as a deflated float."
assert fame_yearly.wages.type().is_float64(), "Wages is not correctly typed as a deflated float."

print("✅ Deflation structural check passed.")